# ThreatFort-LLM: Self-Evolving Classifier Trainer (Google Colab)
Fine-tuning **Llama 3.2 3B Instruct** for safety classification using QLoRA.

**Runtime:** Set Runtime > Change runtime type > **T4 GPU**.

In [ ]:
!pip install -q transformers==4.46.1 datasets==3.0.1 peft==0.13.0 "bitsandbytes>=0.45.0" trl==0.12.0 accelerate==0.34.2 huggingface_hub==0.26.2
!pip uninstall -y torchvision torchaudio 2>/dev/null || true

In [ ]:
import torch, os, json, time, shutil
from huggingface_hub import login
from google.colab import userdata

assert torch.cuda.is_available(), "No GPU. Set Runtime > Change runtime type > T4 GPU."
print(f"GPU : {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

def read_hf_token():
    if os.path.exists('.env'):
        with open('.env') as f:
            for line in f:
                if line.startswith('HF_TOKEN='):
                    val = line.split('=', 1)[1].strip().strip('\"').strip("'")
                    if val and val != 'your_huggingface_token_here':
                        return val
    token = userdata.get('HF_TOKEN')
    if not token:
        raise RuntimeError('HF_TOKEN must be provided in .env or Colab Secrets.')
    return token

login(token=read_hf_token())

In [ ]:
import subprocess

MODEL_ID = 'meta-llama/Llama-3.2-3B-Instruct'
REPO_URL = 'https://github.com/Jaypatil588/robustbench-llm.git'
REPO_BRANCH = 'main'
CLONE_DIR = '/content/threatfort_repo'
REPO_DATA_DIR = os.path.join(CLONE_DIR, 'newDataset', 'processed')
RUNTIME_DATA_DIR = '/content/threatfort_runtime/processed'

if os.path.exists(CLONE_DIR):
    shutil.rmtree(CLONE_DIR)

subprocess.run([
    'git', 'clone', '--branch', REPO_BRANCH, '--single-branch', REPO_URL, CLONE_DIR
], check=True)

required_dataset_files = ('train.jsonl', 'val.jsonl', 'test.jsonl')
for filename in required_dataset_files:
    src = os.path.join(REPO_DATA_DIR, filename)
    if not os.path.exists(src):
        raise FileNotFoundError(f'Required cloned dataset file not found: {src}')

os.makedirs(RUNTIME_DATA_DIR, exist_ok=True)
for filename in required_dataset_files:
    src = os.path.join(REPO_DATA_DIR, filename)
    dst = os.path.join(RUNTIME_DATA_DIR, filename)
    shutil.copy2(src, dst)
    print(f'Copied {src} -> {dst}')

TRAIN_PATH = os.path.join(RUNTIME_DATA_DIR, 'train.jsonl')
VAL_PATH = os.path.join(RUNTIME_DATA_DIR, 'val.jsonl')
TEST_PATH = os.path.join(RUNTIME_DATA_DIR, 'test.jsonl')

print(f'Using training data from {TRAIN_PATH}')
print(f'Using validation data from {VAL_PATH}')
print(f'Copied test data to {TEST_PATH}')

In [ ]:
from datasets import Dataset

def load_and_format(path):
    samples = []
    with open(path) as f:
        for line in f:
            item = json.loads(line)
            missing = {'prompt', 'label'} - set(item)
            if missing:
                raise KeyError(f'{path} row missing required field(s): {sorted(missing)}')
            if item['label'] not in {'adversarial', 'benign'}:
                raise ValueError(f"{path} has invalid label: {item['label']}")
            text = (
                "System: You are a security classifier. "
                "Answer with just 'adversarial' or 'benign'.\n"
                f"User: {item['prompt'][:800]}\n"
                f"Assistant: {item['label']}"
            )
            samples.append({'text': text})
    return Dataset.from_list(samples)

train_dataset = load_and_format(TRAIN_PATH)
val_dataset = load_and_format(VAL_PATH)
print(f'Dataset loaded. Train: {len(train_dataset)}, Val: {len(val_dataset)}')

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
    torch_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

print('Llama 3.2 3B base model loaded in 4-bit precision.')

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules='all-linear',
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
)

model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, lora_config)

trainable, total = model.get_nb_trainable_parameters()
print(f'Trainable Parameters: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)')

In [ ]:
from trl import SFTTrainer, SFTConfig
from transformers import EarlyStoppingCallback

print('Starting QLoRA training with early stopping...')
start = time.time()

training_args = SFTConfig(
    output_dir='./classifier_output',
    max_seq_length=512,
    num_train_epochs=5,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    warmup_ratio=0.03,
    lr_scheduler_type='cosine',
    weight_decay=0.01,
    fp16=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False},
    dataset_text_field='text',
    report_to='none',
    save_strategy='epoch',
    eval_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    save_total_limit=1,
    logging_steps=10,
    seed=42,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=1)],
)

trainer.train()
print(f'Fine-tuning completed in {(time.time()-start)/60:.1f} minutes.')

In [ ]:
from google.colab import files

output_dir = './classifier_adapter'
model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)

shutil.make_archive('classifier_adapter', 'zip', output_dir)
print('Adapter zipped. Downloading classifier_adapter.zip...')
files.download('classifier_adapter.zip')